# Initialisation

In [97]:
from pathlib import Path
import os

# Get current directory
current_dir = os.getcwd() 

# Navigate up from notebooks/ to project root
PROJECT_ROOT = Path(current_dir).resolve().parent

# File paths
DATA_PATH = PROJECT_ROOT
MODEL_PATH = PROJECT_ROOT / "scouting_models.pkl"
SCALER_PATH = PROJECT_ROOT / "tactical_scalers.pkl"

In [98]:
# Library installation 
!pip install statsbombpy pandas mplsoccer 

In [99]:
# Import libraries
from statsbombpy import sb # All the data comes from here
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import pandas as pd
import numpy as np
import math

# Prevent warnings of known issues from flooding output #
import warnings

# Mute the fragmented dataframe and future warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

https://live-data-api-guide.statsbomb.com/api-reference/event-descriptions.html (Event data descriptions)

# List all available seasons

In [102]:
# Get the master list of competitions
comps = sb.competitions()

# Create a list to store the data
summary_data = []

print("Fetching match counts")

# Loop through and count matches for each entry
for index, row in comps.iterrows():
    try:
        # Fetch matches for this specific competition and season
        matches = sb.matches(competition_id=row["competition_id"], season_id=row["season_id"])
        match_count = len(matches)
        
        summary_data.append({
            "Competition": row["competition_name"],
            "Season": row["season_name"],
            "Match Count": match_count,
            "Comp ID": row["competition_id"],
            "Season ID": row["season_id"]
        })
    except Exception as e:
        continue

# Display the results in a tablesummary_df = pd.DataFrame(summary_data)
summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.sort_values("Match Count", ascending=False)

print("\n--- StatsBomb Open Data Availability ---")
print(summary_df.to_string(index=False))

Fetching match counts

--- StatsBomb Open Data Availability ---
            Competition    Season  Match Count  Comp ID  Season ID
                Serie A 2015/2016          380       12         27
                La Liga 2015/2016          380       11         27
         Premier League 2015/2016          380        2         27
                Ligue 1 2015/2016          377        7         27
                 Liga F 2023/2024          240      182        281
                   NWSL      2023          137       49        107
      Frauen Bundesliga 2023/2024          132      135        281
FA Women's Super League 2023/2024          132       37        281
FA Women's Super League 2020/2021          131       37         90
          Serie A Women 2023/2024          130      131        281
    Indian Super league 2021/2022          115     1238        108
FA Women's Super League 2018/2019          107       37          4
FA Women's Super League 2019/2020           87       37         4

# Data Aggregation

In [104]:
# Selected competitions to gather data from
selected_ids = [
    # 2015/2016 Seasons since they have the most data
    {"competition_id": 12, "season_id": 27},    # Serie A 2015/16 (380 Matches)
    {"competition_id": 2, "season_id": 27},     # Premier League 2015/16 (380 Matches)
    {"competition_id": 11, "season_id": 27},    # La Liga 2015/16 (380 Matches)
    {"competition_id": 7, "season_id": 27},     # Ligue 1 2015/16 (377 Matches)
    {"competition_id": 9, "season_id": 27},     # Bundesliga 2015/16 (306 Matches)

    # More "Modern Seasons" with relatively substantial data
    {"competition_id": 43, "season_id": 3},     # World Cup 2018 (64 Matches)
    {"competition_id": 43, "season_id": 106},   # World Cup 2022 (64 Matches)
    {"competition_id": 1267, "season_id": 107}, # AFCON 2023 (52 Matches)
    {"competition_id": 55, "season_id": 43},    # Euro 2020 (51 Matches)
    {"competition_id": 55, "season_id": 282},   # Euro 2024 (51 Matches)
    {"competition_id": 223, "season_id": 282},  # Copa America 2024 (32 Matches)
    {"competition_id": 1238, "season_id": 108}, # Indian Super League 2021/22 (115 Matches)

    # Seasons that only contain one teams data #

    # Pep Guardiola's Barcelona (Messi database)
    {"competition_id": 11, "season_id": 41},    # La Liga 2008/09 (31 Matches)
    {"competition_id": 11, "season_id": 21},    # La Liga 2009/10 (35 Matches)
    {"competition_id": 11, "season_id": 22},    # La Liga 2010/11 (33 Matches)
    {"competition_id": 11, "season_id": 23},    # La Liga 2011/12 (37 Matches)

    # Tito Vilanova's Barcelona (Messi database)
    {"competition_id": 11, "season_id": 24},    # La Liga 2012/13 (32 Matches)

    # Gerardo Martino's Barcelona (Messi database)
    {"competition_id": 11, "season_id": 25},    # La Liga 2013/14 (31 Matches)

    # Luis Enrique's Barcelona (Messi database)
    {"competition_id": 11, "season_id": 26},    # La Liga 2014/15 (38 Matches)
    {"competition_id": 11, "season_id": 2},     # La Liga 2016/17 (34 Matches)

    # Ernesto Valverde's Barcelona (Messi database)
    {"competition_id": 11, "season_id": 1},     # La Liga 2017/18 (36 Matches)
    {"competition_id": 11, "season_id": 4},     # La Liga 2018/19 (34 Matches)
    {"competition_id": 11, "season_id": 42},    # La Liga 2019/20 (33 Matches)

    # Quique Setien's/Ronald Koeman's Barcelona (Messi database)
    {"competition_id": 11, "season_id": 90},    # La Liga 2020/21 (35 Matches)

    # Mauricio Pochettino's PSG (Messi database)
    {"competition_id": 7, "season_id": 108},    # Ligue 1 2021/22 (26 Matches)

    # Christophe Galtier's PSG (Messi database)
    {"competition_id": 7, "season_id": 235},    # Ligue 1 2022/23 (32 Matches)

    # Arsène Wenger's Arsenal (Invincibles)
    {"competition_id": 2, "season_id": 44},     # Premier League 2003/04 (38 Matches)

    # Xabi Alonso's Bayer Leverkusen (Invincibles)
    {"competition_id": 9, "season_id": 281}     # Bundesliga 2023/24 (34 Matches)
]

# Feature Engineering

In [106]:
# Calculates a weight based on a team's possession during a match
def add_padj_weights(events_df):
    # Store every pass event
    passes = events_df[events_df["type"] == "Pass"].copy()

    # Count passes per team per match
    pass_counts = passes.groupby(["match_id", "team"]).size().reset_index(name="passes")
    
    # Count total passes per match
    match_totals = pass_counts.groupby("match_id")["passes"].sum().reset_index(name="total_passes")

    # Calculate possession %
    possession_df = pass_counts.merge(match_totals, on="match_id")
    possession_df["team_possession_pct"] = (possession_df["passes"] / possession_df["total_passes"]) * 100
    possession_df["opp_possession_pct"] = 100 - possession_df["team_possession_pct"]

    # Calculate possession adjusted (PAdj) weight (Statsbomb Forumla)
    # Re-arragned from https://support.hudl.com/s/article/possession?language=en_US&topic=Statsbomb_Global_Football_Data_Glossary
    possession_df["padj_weight"] = 2 / (1 + np.exp(-0.1 * (possession_df["team_possession_pct"] - 50)))

    # Merge weight back into the main events dataframe
    events_with_weights = events_df.merge(
        possession_df[["match_id", "team", "padj_weight"]],
        on=["match_id", "team"],
        how="left"
    )

    # Fill in any missing values with 1.0 (no adjustment)
    events_with_weights["padj_weight"] = events_with_weights["padj_weight"].fillna(1.0)

    return events_with_weights

In [107]:
# Extracts start and end co-ordinates into dedicated columns
def extract_coordinates(events_df):
    # StatsBomb stores location as a list [x, y]
    # This seperates them for use in vector math
    events_df["x_start"] = events_df["location"].apply(lambda x: x[0] if isinstance(x, list) else np.nan)
    events_df["y_start"] = events_df["location"].apply(lambda x: x[1] if isinstance(x, list) else np.nan)
    
    # Initialise empty columns for the end of an action
    events_df["x_end"] = np.nan
    events_df["y_end"] = np.nan
    
    # Fill pass end locations
    mask_pass = events_df["type"] == "Pass"
    events_df.loc[mask_pass, "x_end"] = events_df.loc[mask_pass, "pass_end_location"].apply(lambda x: x[0] if isinstance(x, list) else np.nan)
    events_df.loc[mask_pass, "y_end"] = events_df.loc[mask_pass, "pass_end_location"].apply(lambda x: x[1] if isinstance(x, list) else np.nan)
    
    # Fill carry end locations
    mask_carry = events_df["type"] == "Carry"
    events_df.loc[mask_carry, "x_end"] = events_df.loc[mask_carry, "carry_end_location"].apply(lambda x: x[0] if isinstance(x, list) else np.nan)
    events_df.loc[mask_carry, "y_end"] = events_df.loc[mask_carry, "carry_end_location"].apply(lambda x: x[1] if isinstance(x, list) else np.nan)

    return events_df

#### Possession Features

In [109]:
# Engineers stats that players of possession based teams likely excel in 
def calculate_possession_features(events_df):
    ###############################
    # FEATURE 1: Pass Completion %
    ##############################
    # What percentage of a players passess successfully find a teammate 

    # Store every pass event
    passes = events_df[events_df["type"] == "Pass"].copy()
    
    # 'pass_outcome' is NaN if successful, 'Incomplete'/'Out' etc if failed
    passes["is_successful"] = passes["pass_outcome"].isna()

    # Groups the data by player
    pass_stats = passes.groupby("player").agg(
        total_passes=("id", "count"),              # Number of total passes
        successful_passes=("is_successful", "sum") # Number of successful passes
    )

    # Calculate pass completion percent
    pass_stats["pass_completion_pct"] = pass_stats["successful_passes"] / pass_stats["total_passes"]

    ####################################
    # FEATURE 2: Passes into Final Third
    ####################################
    # Passes that start X < 80 (Final Third line) AND End X >= 80

    # Defines a 'successful final third pass' 
    final_third_mask = (passes["x_start"] < 80) & (passes["x_end"] >= 80) & (passes["is_successful"])

    # Groups the data by player
    final_third_stats = passes[final_third_mask].groupby("player").agg(
        passes_into_final_third=("id", "count") # Number of passes into the final third
    )

    ################################
    # FEATURE 3: Progressive Carries
    ################################
    # Carries that move the ball towards the opponent's goal line at least 10 yards
    # Pitch is 120 yards long. Center of goal is (120, 40)

    # Store ever carry event
    carries = events_df[events_df["type"] == "Carry"].copy()
    
    # Calculate distance to goal (Pythagoras) for Start and End
    start_dist_to_goal = np.sqrt((120 - carries["x_start"])**2 + (40 - carries["y_start"])**2)
    end_dist_to_goal = np.sqrt((120 - carries["x_end"])**2 + (40 - carries["y_end"])**2)
    
    # Progressive if we are 10 yards closer at the end than the start
    carries["is_progressive"] = (start_dist_to_goal - end_dist_to_goal) >= 10

    # Group data by player
    carry_stats = carries[carries["is_progressive"]].groupby("player").agg(
        progressive_carries=("id", "count") # Number of progressive carries
    )

    #######################################
    # FEATURE 4: Receiving % Under Pressure
    #######################################
    # How often does a player successfully recieve the ball when under pressure from an opposition player
    
    # Find any 'Ball Receipt' event where 'under_pressure' is True
    receipts = events_df[events_df["type"].str.contains("Ball Receipt", na=False)].copy()
    
    # Ensure 'under_pressure' column exists (it might be missing if no event in dataset had pressure)
    if "under_pressure" not in receipts.columns:
        receipts["under_pressure"] = False
    
    # Filter for only pressured receipts
    pressured_receipts = receipts[receipts["under_pressure"] == True].copy()
    
    # Outcome: NaN is Complete, anything else is incomplete
    pressured_receipts["is_successful"] = pressured_receipts["ball_receipt_outcome"].isna()

    # Group data by player
    pressure_stats = pressured_receipts.groupby("player").agg(
        total_pressured_receipts=("id", "count"),              # Number of total receipts under pressure
        successful_pressured_receipts=("is_successful", "sum") # Number of successful receipts under pressure
    )
    
    # Avoid division by zero
    pressure_stats["pressure_receipt_pct"] = pressure_stats.apply(
        lambda row: row["successful_pressured_receipts"] / row["total_pressured_receipts"] 
        if row["total_pressured_receipts"] > 0 else 0, axis=1
    )

    #############################
    # Merge Everything into one
    #############################
    features = pd.concat([pass_stats, final_third_stats, carry_stats, pressure_stats], axis=1)
    
    # Fill NaN with 0 (e.g. if a player had passes but 0 progressive carries, it will appear as NaN)
    features = features.fillna(0)
    
    return features

#### High Pressing Features

In [111]:
# Engineers stats that players of high pressing teams likely excel in 
def calculate_pressing_features(events_df):
    ###########################################
    # FEATURE 1: Pressures (Total & Final Third)
    ###########################################
    # The 'pressure' event records when a player closes down an opponent, even without winning the ball back

    # Store all 'pressure' events
    pressures = events_df[events_df["type"] == "Pressure"].copy()
    
    # Final Third Pressures = x_start >= 80 (pitch is 120 long)
    pressures["is_final_third"] = pressures["x_start"] >= 80

    # If a pressure is in the final third, set to the padj weight for the player's team
    pressures["final_third_weight"] = np.where(pressures["is_final_third"], pressures["padj_weight"], 0)

    # Group data by player
    # Instead of totalling the individual actions (1), the possession adjusted weight is used (eg could be 1.5 if a team has more than 50% possession)
    pressure_stats = pressures.groupby("player").agg(
        total_pressures=("padj_weight", "sum"),             # Number of padj pressures
        final_third_pressures=("final_third_weight", "sum") # Number of padj final third pressures
    )
    
    #############################
    # FEATURE 2: Counter-Pressing
    #############################
    # Pressures that happen within 5 seconds of losing the ball
    
    # Check if column exists (it might not if no counterpresses happened in the dataset)
    if "counterpress" in pressures.columns:
        pressures["is_counterpress"] = pressures["counterpress"] == True
    else:
        pressures["is_counterpress"] = False

    # If a pressure event is a counterpress, set to the padj weight for the player's team
    pressures["counterpress_weight"] = np.where(pressures["is_counterpress"], pressures["padj_weight"], 0)

    # Group data by player
    # Same logic as previous
    counterpress_stats = pressures.groupby("player").agg(
        counterpressures=("counterpress_weight", "sum")   # Number of counterpressures
    )
    
    ###########################
    # FEATURE 3: High Turnovers
    ###########################
    # Ball recoveries (winning the ball back from the opposition) that occur within the attacking half (x > 60)

    # Store all ball recovery events
    recoveries = events_df[events_df["type"] == "Ball Recovery"].copy()
    
    # Attacking half recoveries (x > 60)
    recoveries["is_attacking_half"] = recoveries["x_start"] > 60

    # Apply padj weighting as previously explained
    recoveries["high_recovery_weight"] = np.where(recoveries["is_attacking_half"], recoveries["padj_weight"], 0)

    # Group data by player
    # Same logic as previous
    recovery_stats = recoveries.groupby("player").agg(
        high_ball_recoveries=("high_recovery_weight", "sum") # Number of padj high ball recoveries
    )
    
    ################################
    # FEATURE 4: Defensive Distance
    ###############################
    # The average depth of a defensive action (tackle, duel, etc)

    # Store every defensive action
    defensive_actions = events_df[events_df["type"].isin(["Duel", "Interception", "Clearance", "Tackle"])].copy()

    # Group data by player
    # No padj for this statistic as possession shoulnd't have a major effect
    depth_stats = defensive_actions.groupby("player").agg(
        avg_defensive_distance=("x_start", "mean") # Calculate the average distance
    )

    ############################
    # Merge everything into one
    ############################
    
    # Join all separate stats tables
    features = pd.concat([pressure_stats, counterpress_stats, recovery_stats, depth_stats], axis=1)
    
    # Fill NaNs with 0
    features = features.fillna(0)
    
    return features

#### Counterattacking Features

In [113]:
# Engineers stats that players of counter attacking teams likely excel in 
def calculate_counter_features(events_df):
    ########################
    # FEATURE 1: Long Balls
    ########################
    # Passes longer than 30 yards

    # Store all pass events
    passes = events_df[events_df["type"] == "Pass"].copy()
    
    # Calculate Euclidean distance of the pass
    passes["pass_length"] = np.sqrt(
        (passes["x_end"] - passes["x_start"])**2 + 
        (passes["y_end"] - passes["y_start"])**2
    )
    
    # Filter for Long Passes (>30 yards) that were successful
    passes["is_long_pass"] = (passes["pass_length"] > 30) & (passes["pass_outcome"].isna())

    # Group data by player
    long_pass_stats = passes.groupby("player").agg(
        successful_long_passes=("is_long_pass", "sum") # Number of successful long passes
    )
    
    ############################
    # FEATURE 2: Pass Directness
    ############################
    # Ratio of forward pass distance against total pass distance
    
    # Forward distance = x_end - x_start
    passes["forward_dist"] = passes["x_end"] - passes["x_start"]
    passes["forward_dist"] = passes["forward_dist"].clip(lower=0) # Turn negatives (back passes) to 0

    # Group data by player
    directness_stats = passes.groupby("player").agg(
        total_pass_distance=("pass_length", "sum"),    # Total pass distance
        total_forward_distance=("forward_dist", "sum") # Total forward distance
    )
    
    # Calculate Ratio (Forward % of Total)
    directness_stats["pass_directness_ratio"] = (
        directness_stats["total_forward_distance"] / directness_stats["total_pass_distance"]
    ).fillna(0)

    ##############################
    # FEATURE 3: Explosive Carries
    ##############################
    # Carries that are over 15 yards in any direction

    # Store all carry events
    carries = events_df[events_df["type"] == "Carry"].copy()

    # Calculate Euclidean distance of the carry
    carries["carry_length"] = np.sqrt(
        (carries["x_end"] - carries["x_start"])**2 + 
        (carries["y_end"] - carries["y_start"])**2
    )

    # Define an explosive carry
    carries["is_explosive"] = carries["carry_length"] > 15

    # Group data by player
    carry_stats = carries.groupby("player").agg(
        explosive_carries=("is_explosive", "sum") # Number of explosive carries
    )

    ############################
    # FEATURE 4: Shot Efficiency
    ############################
    # xG per shot
    # xG (Expceted Goals) is a model that predicts the probability that a given shot will be a goal

    # Store all shot events
    shots = events_df[events_df["type"] == "Shot"].copy()
    
    # xG is stored in the 'shot' object
    # Check that a shots xG value is present
    if "shot_statsbomb_xg" not in shots.columns:
        # If column is missing
        shots["shot_statsbomb_xg"] = 0

    # Group data by player
    xg_stats = shots.groupby("player").agg(
        total_shots=("id", "count"),          # Number of shots
        total_xg=("shot_statsbomb_xg", "sum") # Number of xG
    )

    # Calculate average xG (total xG/total shots)
    xg_stats["avg_shot_quality"] = (xg_stats["total_xg"] / xg_stats["total_shots"]).fillna(0)
    
    ##########################
    # Merge everythng into one
    ##########################

    # Join all seperate stats
    features = pd.concat([long_pass_stats, directness_stats, carry_stats, xg_stats], axis=1)

    # Fill empty stats
    features = features.fillna(0)
    
    return features

#### Extra attacker features

In [115]:
# Engineers specific metrics to distinguish attacker profiles
def calculate_attacker_features(events_df):
    #######################################
    # FEATURE 1: Touches in Opposition Box
    #######################################
    # How many touches a player makes in the oppositions penalty box (x >= 102), (18 <= y <= 62)

    # Store all 'on the ball' events that count as touches
    on_ball_events = events_df[events_df["type"].isin(["Pass", "Carry", "Shot", "Ball Receipt*"])].copy()

    # Calculate whether said events are within the opponents penalty area
    on_ball_events["in_opp_box"] = (
        (on_ball_events["x_start"] >= 102) &
        (on_ball_events["y_start"] >= 18) &
        (on_ball_events["y_start"] <=62)
    )

    # Group data by player
    box_touches_stats = on_ball_events.groupby("player").agg(touches_in_box=("in_opp_box", "sum")) # Number of touches in the oppositions box

    ##############################
    # FEATURE 2: Aerial Duels Won
    ##############################
    # Aerial duals are usually when a player from both teams goes to head the ball at the same time
    # Winning an aerial dual usually means a player got to the ball first

    # List possible 'aerial_won' columns
    possible_aerial_cols = [
        "pass_aerial_won", 
        "clearance_aerial_won", 
        "shot_aerial_won", 
        "miscontrol_aerial_won"
    ]

    # Identify which 'aerial_won' columns exist in this specific DataFrame
    existing_aerial_cols = [col for col in possible_aerial_cols if col in events_df.columns]
    
    # Check if any of those columns are True for a given event
    if existing_aerial_cols:
        events_df["is_aerial_won"] = events_df[existing_aerial_cols].fillna(False).any(axis=1) 
    else:
        events_df["is_aerial_won"] = False
        
    # Group data by player
    aerial_stats = events_df[events_df["is_aerial_won"]].groupby("player").agg(
        aerial_duels_won=("id", "count") # Number of aerial duals won
    )

    ########################################
    # FEATURE 3: Average Distance From Goal
    ########################################
    # How far from the opponents goal does a player recieve the ball on average

    # Store every 'ball receipt' event
    receipts = events_df[events_df["type"].str.contains("Ball Receipt", na=False)].copy()
    
    # Calculate Pythagoras distance from the center of the opponent's goal (120, 40)
    receipts["dist_from_goal"] = np.sqrt((120 - receipts["x_start"])**2 + (40 - receipts["y_start"])**2)

    # Group data by player
    dist_stats = receipts.groupby("player").agg(
        avg_receipt_distance_from_goal=("dist_from_goal", "mean") # Calculate average (mean) distance of ball receipt from goal
    )

    ############################
    # Merge everything into one
    ############################

    # Join all seperate stats
    features = pd.concat([box_touches_stats, aerial_stats, dist_stats], axis=1)

    # Fill empty stats
    features = features.fillna(0)
    
    return features

# Data Normalisation and final dataset creation

In [117]:
# Calculate how many total minutes each player has played 
def calculate_minutes_played(events_df):
    # Empty dictionary to store minutes played 
    player_minutes = {}
    
    # Group by individual match to handle each game separately
    for match_id, match_events in events_df.groupby("match_id"):
        
        # Determine match length by looking for the maximum minute in the events dataframe
        max_minute = match_events["minute"].max()
        
        # Process the starting XI (which players started the match)
        starting_xi_events = match_events[match_events["type"] == "Starting XI"]

        # Loops through both teams
        for _, row in starting_xi_events.iterrows():
            # Starting lineup is stored inside a nested JSON directory under a column called 'tactics'
            tactics = row.get("tactics")

            # Ensure lineup is present
            if isinstance(tactics, dict) and "lineup" in tactics:
                for player_entry in tactics["lineup"]:
                    # Extract player name
                    p_name = player_entry["player"]["name"]

                    # Assume maximum minutes played
                    # If a player does not exist within player_minutes, initialise to 0 and add max minutes
                    # If a player already exists within player_minutes, get their current total and add max minutes to it
                    player_minutes[p_name] = player_minutes.get(p_name, 0) + max_minute

        # Process substitutions # 
        # Store every substitution event in the match
        subs = match_events[match_events["type"] == "Substitution"]

        # Loop through each substitution event
        for _, row in subs.iterrows():
            player_off = row["player"] # Player leaving the pitch
            player_on = row.get("substitution_replacement", None) # Player entering the pitch
            sub_minute = row["minute"] # Minute the substitution occured
            
            # Subtract unplayed minutes from the starter (Player off)
            if player_off in player_minutes:
                # Remove the minutes they missed after being subbed from their total
                player_minutes[player_off] -= (max_minute - sub_minute)
            
            # Add minutes for the sub (Player on)
            if player_on:
                # Checks if the sub is already within player_minutes, if not, initialises to 0
                # Adds the minutes they played to their running total
                player_minutes[player_on] = player_minutes.get(player_on, 0) + (max_minute - sub_minute)

    # Return the python dictionary as a pandas series for easy merging with the rest of the data
    return pd.Series(player_minutes, name="minutes_played")


# Combine it all together
def build_final_dataset(events_df):
    events_df = add_padj_weights(events_df) # Add possession weight to each player
    events_df = extract_coordinates(events_df) # Extract split event coordinates 
    
    # Feature Engineering
    possession = calculate_possession_features(events_df)
    pressing = calculate_pressing_features(events_df)
    counter = calculate_counter_features(events_df)
    attacker_metrics = calculate_attacker_features(events_df)
    
    # Merge all data
    total_stats = pd.concat([possession, pressing, counter, attacker_metrics], axis=1).fillna(0)
    
    # Calculate minutes played
    minutes = calculate_minutes_played(events_df)

    # Generate a final dataframe by merging minutes played with the rest of the data
    final_df = total_stats.join(minutes, how="inner")

    ######################
    # Normalize to Per 90
    #####################
    # Some stats should be normalised to 'per 90 minutes' to allow for fairer comparison of players independant of total minutes played
    # A player with more minutes may appear to have better stats than another but may actually perform worse game by game
    
    # List of columns to normalize (excluding ratios and percentages)
    cols_to_normalize = [
        "total_passes", "successful_passes", "passes_into_final_third",
        "progressive_carries", "total_pressured_receipts", "successful_pressured_receipts",
        "total_pressures", "final_third_pressures", "counterpressures", 
        "high_ball_recoveries", "successful_long_passes", "total_pass_distance", "total_forward_distance", 
        "explosive_carries", "total_shots", "total_xg", "touches_in_box", "aerial_duels_won"
    ]

    # Loop through each column and normalise to 'per 90 minutes'
    for col in cols_to_normalize:
        if col in final_df.columns:
            final_df[f'{col}_per90'] = (final_df[col] / final_df["minutes_played"]) * 90
            
    # Filter out players with low minute as that can skew the data
    # eg a player who has only played 10 minutes but scored would have an absurd and unrealistic goals per 90 ratio
    final_df = final_df[final_df["minutes_played"] > 300]

    # Check size of stats before merging
    print(f"Count of players with Event Stats: {total_stats.shape[0]}")
    
    # Check size of minutes calculated
    minutes = calculate_minutes_played(events_df)
    print(f"Count of players with Minutes calculated: {len(minutes)}")
    
    # Check the merge before filtering low minutes
    # Outer ensures everyone is seen
    debug_df = total_stats.join(minutes, how="outer")
    print(f"Count of players after Merge: {debug_df.shape[0]}")
    
    # Check for Missing Minutes
    players_missing_minutes = debug_df["minutes_played"].isna().sum()
    print(f"Players with Stats but no minutes detected: {players_missing_minutes}")
    
    # Check the impact of the filter
    over_300 = debug_df[debug_df["minutes_played"] > 300]
    print(f"Players with > 300 Minutes: {over_300.shape[0]}")

    # Return the final dataframe
    return final_df

### Extract information about teams and positions of players to add to the final dataset

In [119]:
# Extracts the team and primary position for each player
def get_player_metadata(events_df):
    ##############################
    # Get a player's primary team
    ##############################
    # Automatically determines a players primary team by finding the mode (most common) one
    # If one is not found it returns the first team the player was assigned to
    # This covers players who have played in multiple teams
    team_map = events_df.groupby("player")["team"].agg(
        lambda x: pd.Series.mode(x)[0] if not pd.Series.mode(x).empty else x.iloc[0]
    )
    
    #################################
    # Get a player's primary Position
    #################################
    # Automatically determines a players most played position by finding the mode (most common) one
    # If no posiiton is found, it is set to 'unknown'
    # This covers any positional changes a player may have to make
    position_map = events_df.groupby("player")["position"].agg(
        lambda x: pd.Series.mode(x)[0] if not pd.Series.mode(x).empty else "Unknown"
    )
    
    ##########################
    # Combine into a DataFrame
    ##########################
    metadata = pd.concat([team_map, position_map], axis=1)
    metadata.columns = ["team_name", "primary_position"] # Rename columns to be clear and readable
    
    return metadata

# Testing

In [121]:
# Create a fake StatsBomb "Match" to test each feature engineering function
def create_test_data():
    events = []
    match_id = 999
    team_a = "Red Team"
    team_b = "Blue Team"

    # Start Match
    tactics_a = {"lineup": [{"player": {"name": "Possession Pete"}}, {"player": {"name": "Pressing Paul"}}]}
    tactics_b = {"lineup": [{"player": {"name": "Countering Chris"}}, {"player": {"name": "Attacking Andy"}}]}
    events.append({"id": "s1", "match_id": match_id, "minute": 0, "type": "Starting XI", "team": team_a, "tactics": tactics_a})
    events.append({"id": "s2", "match_id": match_id, "minute": 0, "type": "Starting XI", "team": team_b, "tactics": tactics_b})

    # End Match at 1000 minutes to everyone passes the 300 minutes threshold
    events.append({"id": "end", "match_id": match_id, "minute": 1000, "type": "Half End", "team": team_a})

    # Team A completes 80 passes and Team B completes 20 leading to 80%-20% possession for weight testing
    for i in range(79):
        events.append({"id": f"pA_{i}", "match_id": match_id, "minute": 10, "type": "Pass", "team": team_a, "player": "Possession Pete", "location": [50.0, 40.0], "pass_end_location": [60.0, 40.0], "pass_outcome": np.nan})
    for i in range(19):
        events.append({"id": f"pB_{i}", "match_id": match_id, "minute": 20, "type": "Pass", "team": team_b, "player": "Countering Chris", "location": [30.0, 40.0], "pass_end_location": [70.0, 40.0], "pass_outcome": np.nan}) # Long passes
 
    # Test possession features
    events.append({"id": "pete_c", "match_id": match_id, "minute": 15, "type": "Carry", "team": team_a, "player": "Possession Pete", "location": [60.0, 40.0], "carry_end_location": [71.0, 40.0]}) # 1 Progressive Carry
    events.append({"id": "pete_r", "match_id": match_id, "minute": 16, "type": "Ball Receipt*", "team": team_a, "player": "Possession Pete", "location": [60.0, 40.0], "under_pressure": True, "ball_receipt_outcome": np.nan}) # 1 Completed Pressured Receipt
    events.append({"id": "pete_r2", "match_id": match_id, "minute": 16, "type": "Ball Receipt*", "team": team_a, "player": "Possession Pete", "location": [60.0, 40.0], "under_pressure": True, "ball_receipt_outcome": "Incomplete"}) # 1 failed Pressured Receipt
    events.append({"id": "pete_p", "match_id": match_id, "minute": 17, "type": "Pass", "team": team_a, "player": "Possession Pete", "location": [70.0, 40.0], "pass_end_location": [85.0, 40.0], "pass_outcome": np.nan}) # 1 Final Third Pass

    # Test pressing and defensive features
    events.append({"id": "paul_p1", "match_id": match_id, "minute": 25, "type": "Pressure", "team": team_a, "player": "Pressing Paul", "location": [85.0, 40.0], "counterpress": True}) # Final Third + Counterpress
    events.append({"id": "paul_p2", "match_id": match_id, "minute": 26, "type": "Pressure", "team": team_a, "player": "Pressing Paul", "location": [90.0, 40.0], "counterpress": False}) # Final third
    events.append({"id": "chris_p1", "match_id": match_id, "minute": 48, "type": "Pressure", "team": team_b, "player": "Countering Chris", "location": [40.0, 40.0], "counterpress": False}) # Pressure for 20% possession team to ensure weighting is working, also not final third
    events.append({"id": "paul_br", "match_id": match_id, "minute": 27, "type": "Ball Recovery", "team": team_a, "player": "Pressing Paul", "location": [85.0, 40.0]}) # High ball recovery
    events.append({"id": "paul_d",  "match_id": match_id, "minute": 28, "type": "Duel", "team": team_a, "player": "Pressing Paul", "location": [40.0, 40.0], "duel_type": "Tackle"}) # Depth distance test

    # Test counterattacking features
    events.append({"id": "chris_c", "match_id": match_id, "minute": 35, "type": "Carry", "team": team_b, "player": "Countering Chris", "location": [50.0, 40.0], "carry_end_location": [50.0, 60.0]}) # Explosive Carry (>15 yards), horizontal so not progressive
    events.append({"id": "chris_s", "match_id": match_id, "minute": 36, "type": "Shot", "team": team_b, "player": "Countering Chris", "location": [105.0, 40.0], "shot_statsbomb_xg": 0.45}) # Shot xG
    events.append({"id": "chris_s2", "match_id": match_id, "minute": 36, "type": "Shot", "team": team_b, "player": "Countering Chris", "location": [105.0, 40.0], "shot_statsbomb_xg": 0.11}) # Shot xG

    # Test extra attacking features
    events.append({"id": "andy_c", "match_id": match_id, "minute": 45, "type": "Carry", "team": team_b, "player": "Attacking Andy", "location": [105.0, 40.0], "carry_end_location": [110.0, 40.0]}) # Touch in opposition box
    events.append({"id": "andy_p", "match_id": match_id, "minute": 46, "type": "Pass", "team": team_b, "player": "Attacking Andy", "location": [100.0, 40.0], "pass_end_location": [90.0, 40.0], "pass_aerial_won": True, "pass_outcome": "Incomplete"}) # Aerial duel won (pass_aerial_won so counts as pass)
    events.append({"id": "andy_r", "match_id": match_id, "minute": 47, "type": "Ball Receipt*", "team": team_b, "player": "Attacking Andy", "location": [100.0, 40.0]}) # Receipt distance from goal (X:120, Y:40) = 20 yards

    # Make a substitution to test minute calculations
    events.append({
        "id": "sub_1", "match_id": match_id, "minute": 500, "type": "Substitution", 
        "team": team_b, "player": "Attacking Andy", "substitution_replacement": "Subbed-on Steve"
    })

    # Give the substitute player an action to test minutes
    events.append({"id": "steve_c", "match_id": match_id, "minute": 200, "type": "Carry", "team": team_b, "player": "Subbed-on Steve", "location": [105.0, 40.0], "carry_end_location": [110.0, 40.0]}) # Touch in box

    # Compile the dataframe
    df = pd.DataFrame(events)

    # Add missing StatsBomb columns used in the functions to avoid KeyErrors
    for col in ['pass_outcome', 'under_pressure', 'ball_receipt_outcome', 'counterpress', 'pass_aerial_won', 'clearance_aerial_won', 'shot_aerial_won', 'miscontrol_aerial_won', 'shot_statsbomb_xg', 'player', 'team', 'position', 'substitution_replacement', 'tactics', 'duel_type']:
        if col not in df.columns:
            df[col] = np.nan
            
    return df

In [122]:
# Run the test function
print("Generating Dummy Events")
dummy_events = create_test_data()

print("Passing through features engineering pipeline")
test_results_df = build_final_dataset(dummy_events)

# Display the results
print(test_results_df)
print(test_results_df["total_passes"][3])

# Assert that data is correct #

team_a_possession = 0.8
team_b_possession = 0.2

# total_passes
assert test_results_df["total_passes"][0] == 1 # Andy's aerial dual counts as a pass
assert test_results_df["total_passes"][1] == 19 # Chris
assert test_results_df["total_passes"][2] == 80 # Pete
assert test_results_df["total_passes"][3] == 0  # Paul
assert test_results_df["total_passes"][4] == 0  # Steve

# successful_passes 
assert test_results_df["successful_passes"][0] == 0 # Andy's aerial dual pass was failed :(
assert test_results_df["successful_passes"][1] == 19 # Chris
assert test_results_df["successful_passes"][2] == 80 # Pete
assert test_results_df["successful_passes"][3] == 0  # Paul
assert test_results_df["successful_passes"][4] == 0  # Steve

# pass_completion_pct
assert test_results_df["pass_completion_pct"][0] == 0 # Andy
assert test_results_df["pass_completion_pct"][1] == 1 # Chris
assert test_results_df["pass_completion_pct"][2] == 1 # Pete
assert test_results_df["pass_completion_pct"][3] == 0 # Paul
assert test_results_df["pass_completion_pct"][4] == 0 # Steve

# passes_into_final_third
assert test_results_df["passes_into_final_third"][0] == 0 # Andy
assert test_results_df["passes_into_final_third"][1] == 0 # Chris
assert test_results_df["passes_into_final_third"][2] == 1 # Pete should be the only one
assert test_results_df["passes_into_final_third"][3] == 0 # Paul
assert test_results_df["passes_into_final_third"][4] == 0 # Steve

# progressive_carries
assert test_results_df["progressive_carries"][0] == 0 # Andy
assert test_results_df["progressive_carries"][1] == 0 # Chris
assert test_results_df["progressive_carries"][2] == 1 # Pete should be the only one
assert test_results_df["progressive_carries"][3] == 0 # Paul
assert test_results_df["progressive_carries"][4] == 0 # Steve

# total_pressured_receipts
assert test_results_df["total_pressured_receipts"][0] == 0 # Andy
assert test_results_df["total_pressured_receipts"][1] == 0 # Chris
assert test_results_df["total_pressured_receipts"][2] == 2 # Pete should be the only one
assert test_results_df["total_pressured_receipts"][3] == 0 # Paul 
assert test_results_df["total_pressured_receipts"][4] == 0 # Steve

# successful_pressured_receipts
assert test_results_df["successful_pressured_receipts"][0] == 0 # Andy
assert test_results_df["successful_pressured_receipts"][1] == 0 # Chris
assert test_results_df["successful_pressured_receipts"][2] == 1 # Pete should have failed one and completed one
assert test_results_df["successful_pressured_receipts"][3] == 0 # Paul
assert test_results_df["successful_pressured_receipts"][4] == 0 # Steve

# pressure_receipt_pct
assert test_results_df["pressure_receipt_pct"][0] == 0 # Andy
assert test_results_df["pressure_receipt_pct"][1] == 0 # Chris
assert test_results_df["pressure_receipt_pct"][2] == 0.5 # Pete should have 50%
assert test_results_df["pressure_receipt_pct"][3] == 0 # Paul
assert test_results_df["pressure_receipt_pct"][4] == 0 # Steve

# total_pressures (possession adjusted)
assert test_results_df["total_pressures"][0] == 0 # Andy

pressures = 1 # Chris attempted 1 pressure 
answer = (pressures*2/(1+np.exp(-10*(team_b_possession-0.5))))
assert test_results_df["total_pressures"][1] == answer # Chris


assert test_results_df["total_pressures"][2] == 0 # Pete

pressures = 2 # Paul attempted 2 pressures
answer = (pressures*2/(1+np.exp(-10*(team_a_possession-0.5))))
assert test_results_df["total_pressures"][3] == answer # Paul

assert test_results_df["total_pressures"][4] == 0 # Steve

# final_third_pressures (possession adjusted)
assert test_results_df["final_third_pressures"][0] == 0 # Andy

assert test_results_df["final_third_pressures"][1] == 0 # Chris' pressure was not final third

assert test_results_df["final_third_pressures"][2] == 0 # Pete

pressures = 2 # Paul attempted 2 pressures
answer = (pressures*2/(1+np.exp(-10*(team_a_possession-0.5))))
assert test_results_df["final_third_pressures"][3] == answer # Paul's should be the same as total pressures as all were final third

assert test_results_df["final_third_pressures"][4] == 0 # Steve

# counterpressures (possession adjusted)
assert test_results_df["counterpressures"][0] == 0 # Andy
assert test_results_df["counterpressures"][1] == 0 # Chris
assert test_results_df["counterpressures"][2] == 0 # Pete

counterpressures = 1
answer = (counterpressures*2/(1+np.exp(-10*(team_a_possession-0.5))))
assert test_results_df["counterpressures"][3] == answer # Paul should have 1 (possession adjusted)

assert test_results_df["counterpressures"][4] == 0 # Steve

# high_ball_recoveries (possession adjusted)
assert test_results_df["high_ball_recoveries"][0] == 0 # Andy
assert test_results_df["high_ball_recoveries"][1] == 0 # Chris
assert test_results_df["high_ball_recoveries"][2] == 0 # Pete

high_ball_recoveries = 1
answer = (high_ball_recoveries*2/(1+np.exp(-10*(team_a_possession-0.5))))
assert test_results_df["high_ball_recoveries"][3] == answer # Paul should have 1 (possession adjusted)

assert test_results_df["high_ball_recoveries"][4] == 0 # Steve

# avg_defensive_distance
assert test_results_df["avg_defensive_distance"][0] == 0 # Andy
assert test_results_df["avg_defensive_distance"][1] == 0 # Chris
assert test_results_df["avg_defensive_distance"][2] == 0 # Pete
assert test_results_df["avg_defensive_distance"][3] == 40 # Paul
assert test_results_df["avg_defensive_distance"][4] == 0 # Steve

# successful_long_passes
assert test_results_df["successful_long_passes"][0] == 0  # Andy
assert test_results_df["successful_long_passes"][1] == 19 # Chris
assert test_results_df["successful_long_passes"][2] == 0  # Pete
assert test_results_df["successful_long_passes"][3] == 0  # Paul
assert test_results_df["successful_long_passes"][4] == 0 # Steve

# total_pass_distance
assert test_results_df["total_pass_distance"][0] == 10  # Andy
assert test_results_df["total_pass_distance"][1] == 760 # Chris
assert test_results_df["total_pass_distance"][2] == 805 # Pete
assert test_results_df["total_pass_distance"][3] == 0   # Paul
assert test_results_df["total_pass_distance"][4] == 0 # Steve

# total_forward_distance
assert test_results_df["total_forward_distance"][0] == 0  # Andy's pass was backwards
assert test_results_df["total_forward_distance"][1] == 760 # Chris
assert test_results_df["total_forward_distance"][2] == 805 # Pete
assert test_results_df["total_forward_distance"][3] == 0   # Paul
assert test_results_df["total_forward_distance"][4] == 0 # Steve

# pass_directness_ratio
assert test_results_df["pass_directness_ratio"][0] == 0 # Andy, no passes were forward
assert test_results_df["pass_directness_ratio"][1] == 1 # Chris, every pass was forward
assert test_results_df["pass_directness_ratio"][2] == 1 # Pete, every pass was forward
assert test_results_df["pass_directness_ratio"][3] == 0 # Paul
assert test_results_df["pass_directness_ratio"][4] == 0 # Steve

# explosive_carries
assert test_results_df["explosive_carries"][0] == 0 # Andy
assert test_results_df["explosive_carries"][1] == 1 # Chris should be the only one
assert test_results_df["explosive_carries"][2] == 0 # Pete
assert test_results_df["explosive_carries"][3] == 0 # Paul
assert test_results_df["explosive_carries"][4] == 0 # Steve

# total_shots
assert test_results_df["total_shots"][0] == 0 # Andy
assert test_results_df["total_shots"][1] == 2 # Chris should be the only one
assert test_results_df["total_shots"][2] == 0 # Pete
assert test_results_df["total_shots"][3] == 0 # Paul
assert test_results_df["total_shots"][4] == 0 # Steve

# total_xg
assert test_results_df["total_xg"][0] == 0 # Andy
assert test_results_df["total_xg"][1] == 0.56 # Chris should be the only one
assert test_results_df["total_xg"][2] == 0 # Pete
assert test_results_df["total_xg"][3] == 0 # Paul
assert test_results_df["total_xg"][4] == 0 # Steve

# avg_shot_quality
assert test_results_df["avg_shot_quality"][0] == 0 # Andy
assert test_results_df["avg_shot_quality"][1] == 0.28 # Chris should be the only one
assert test_results_df["avg_shot_quality"][2] == 0 # Pete
assert test_results_df["avg_shot_quality"][3] == 0 # Paul
assert test_results_df["avg_shot_quality"][4] == 0 # Steve

# touches_in_box
assert test_results_df["touches_in_box"][0] == 1 # Andy performed one carry into box
assert test_results_df["touches_in_box"][1] == 2 # Chris took two shots in box
assert test_results_df["touches_in_box"][2] == 0 # Pete
assert test_results_df["touches_in_box"][3] == 0 # Paul
assert test_results_df["touches_in_box"][4] == 1 # Steve had 1

# aerial_duels_won
assert test_results_df["aerial_duels_won"][0] == 1 # Andy should be the only one
assert test_results_df["aerial_duels_won"][1] == 0 # Chris 
assert test_results_df["aerial_duels_won"][2] == 0 # Pete
assert test_results_df["aerial_duels_won"][3] == 0 # Paul
assert test_results_df["aerial_duels_won"][4] == 0 # Steve

# avg_receipt_distance_from_goal
assert test_results_df["avg_receipt_distance_from_goal"][0] == 20 # Andy
assert test_results_df["avg_receipt_distance_from_goal"][1] == 0 # Chris 
assert test_results_df["avg_receipt_distance_from_goal"][2] == 60 # Pete
assert test_results_df["avg_receipt_distance_from_goal"][3] == 0 # Paul
assert test_results_df["avg_receipt_distance_from_goal"][4] == 0 # Steve

# minutes_played
assert test_results_df["minutes_played"][0] == 500 # Andy
assert test_results_df["minutes_played"][1] == 1000 # Chris 
assert test_results_df["minutes_played"][2] == 1000 # Pete
assert test_results_df["minutes_played"][3] == 1000 # Paul
assert test_results_df["minutes_played"][4] == 500 # Steve

# total_passes_per90
assert test_results_df["total_passes_per90"][0] == (1/500)*90 # Andy's aerial dual counts as a pass
assert test_results_df["total_passes_per90"][1] == (19/1000)*90 # Chris
assert test_results_df["total_passes_per90"][2] == (80/1000)*90 # Pete
assert test_results_df["total_passes_per90"][3] == 0  # Paul
assert test_results_df["total_passes_per90"][4] == 0 # Steve

# successful_passes_per90
assert test_results_df["successful_passes_per90"][0] == 0 # Andy's aerial dual pass was failed :(
assert test_results_df["successful_passes_per90"][1] == (19/1000)*90 # Chris
assert test_results_df["successful_passes_per90"][2] == (80/1000)*90 # Pete
assert test_results_df["successful_passes_per90"][3] == 0  # Paul
assert test_results_df["successful_passes_per90"][4] == 0 # Steve

# passes_into_final_third_per90
assert test_results_df["passes_into_final_third_per90"][0] == 0 # Andy
assert test_results_df["passes_into_final_third_per90"][1] == 0 # Chris
assert test_results_df["passes_into_final_third_per90"][2] == (1/1000)*90 # Pete should be the only one
assert test_results_df["passes_into_final_third_per90"][3] == 0 # Paul
assert test_results_df["passes_into_final_third_per90"][4] == 0 # Steve

# progressive_carries_per90
assert test_results_df["progressive_carries_per90"][0] == 0 # Andy
assert test_results_df["progressive_carries_per90"][1] == 0 # Chris
assert test_results_df["progressive_carries_per90"][2] == (1/1000)*90 # Pete should be the only one
assert test_results_df["progressive_carries_per90"][3] == 0 # Paul
assert test_results_df["progressive_carries_per90"][4] == 0 # Steve

# total_pressured_receipts_per90
assert test_results_df["total_pressured_receipts_per90"][0] == 0 # Andy
assert test_results_df["total_pressured_receipts_per90"][1] == 0 # Chris
assert test_results_df["total_pressured_receipts_per90"][2] == (2/1000)*90 # Pete should be the only one
assert test_results_df["total_pressured_receipts_per90"][3] == 0 # Paul 
assert test_results_df["total_pressured_receipts_per90"][4] == 0 # Steve

# successful_pressured_receipts_per90
assert test_results_df["successful_pressured_receipts_per90"][0] == 0 # Andy
assert test_results_df["successful_pressured_receipts_per90"][1] == 0 # Chris
assert test_results_df["successful_pressured_receipts_per90"][2] == (1/1000)*90 # Pete should have failed one and completed one
assert test_results_df["successful_pressured_receipts_per90"][3] == 0 # Paul
assert test_results_df["successful_pressured_receipts_per90"][4] == 0 # Steve

# total_pressures_per90 
assert test_results_df["total_pressures_per90"][0] == 0 # Andy

pressures = 1 # Chris attempted 1 pressure 
answer = (pressures*2/(1+np.exp(-10*(team_b_possession-0.5))))
assert test_results_df["total_pressures_per90"][1] == (answer/1000)*90 # Chris


assert test_results_df["total_pressures_per90"][2] == 0 # Pete

pressures = 2 # Paul attempted 2 pressures
answer = (pressures*2/(1+np.exp(-10*(team_a_possession-0.5))))
assert test_results_df["total_pressures_per90"][3] == (answer/1000)*90 # Paul

assert test_results_df["total_pressures_per90"][4] == 0 # Steve


# final_third_pressures_per90
assert test_results_df["final_third_pressures_per90"][0] == 0 # Andy

assert test_results_df["final_third_pressures_per90"][1] == 0 # Chris' pressure was not final third

assert test_results_df["final_third_pressures_per90"][2] == 0 # Pete

pressures = 2 # Paul attempted 2 pressures
answer = (pressures*2/(1+np.exp(-10*(team_a_possession-0.5))))
assert test_results_df["final_third_pressures_per90"][3] == (answer/1000)*90 # Paul's should be the same as total pressures as all were final third

assert test_results_df["final_third_pressures_per90"][4] == 0 # Steve

# counterpressures_per90
assert test_results_df["counterpressures_per90"][0] == 0 # Andy
assert test_results_df["counterpressures_per90"][1] == 0 # Chris
assert test_results_df["counterpressures_per90"][2] == 0 # Pete

counterpressures = 1
answer = (counterpressures*2/(1+np.exp(-10*(team_a_possession-0.5))))
assert test_results_df["counterpressures_per90"][3] == (answer/1000)*90 # Paul should have 1 (possession adjusted)

assert test_results_df["counterpressures_per90"][4] == 0 # Steve

# high_ball_recoveries_per90
assert test_results_df["high_ball_recoveries_per90"][0] == 0 # Andy
assert test_results_df["high_ball_recoveries_per90"][1] == 0 # Chris
assert test_results_df["high_ball_recoveries_per90"][2] == 0 # Pete

high_ball_recoveries = 1
answer = (high_ball_recoveries*2/(1+np.exp(-10*(team_a_possession-0.5))))
assert test_results_df["high_ball_recoveries_per90"][3] == (answer/1000)*90 # Paul should have 1 (possession adjusted)

assert test_results_df["high_ball_recoveries_per90"][4] == 0 # Steve

# successful_long_passes_per90
assert test_results_df["successful_long_passes_per90"][0] == 0  # Andy
assert test_results_df["successful_long_passes_per90"][1] == (19/1000)*90 # Chris
assert test_results_df["successful_long_passes_per90"][2] == 0  # Pete
assert test_results_df["successful_long_passes_per90"][3] == 0  # Paul
assert test_results_df["successful_long_passes_per90"][4] == 0 # Steve

# total_pass_distance_per90
assert test_results_df["total_pass_distance_per90"][0] == (10/500)*90  # Andy
assert test_results_df["total_pass_distance_per90"][1] == (760/1000)*90 # Chris
assert test_results_df["total_pass_distance_per90"][2] == (805/1000)*90 # Pete
assert test_results_df["total_pass_distance_per90"][3] == 0   # Paul
assert test_results_df["total_pass_distance_per90"][4] == 0 # Steve

# total_forward_distance_per90
assert test_results_df["total_forward_distance_per90"][0] == 0  # Andy's pass was backwards
assert test_results_df["total_forward_distance_per90"][1] == (760/1000)*90 # Chris
assert test_results_df["total_forward_distance_per90"][2] == (805/1000)*90 # Pete
assert test_results_df["total_forward_distance_per90"][3] == 0   # Paul
assert test_results_df["total_forward_distance_per90"][4] == 0 # Steve

# explosive_carries_per90
assert test_results_df["explosive_carries_per90"][0] == 0 # Andy
assert test_results_df["explosive_carries_per90"][1] == (1/1000)*90 # Chris should be the only one
assert test_results_df["explosive_carries_per90"][2] == 0 # Pete
assert test_results_df["explosive_carries_per90"][3] == 0 # Paul
assert test_results_df["explosive_carries_per90"][4] == 0 # Steve

# total_shots_per90
assert test_results_df["total_shots_per90"][0] == 0 # Andy
assert test_results_df["total_shots_per90"][1] == (2/1000)*90 # Chris should be the only one
assert test_results_df["total_shots_per90"][2] == 0 # Pete
assert test_results_df["total_shots_per90"][3] == 0 # Paul
assert test_results_df["total_shots_per90"][4] == 0 # Steve

# total_xg_per90
assert test_results_df["total_xg_per90"][0] == 0 # Andy
assert test_results_df["total_xg_per90"][1] == (0.56/1000)*90 # Chris should be the only one
assert test_results_df["total_xg_per90"][2] == 0 # Pete
assert test_results_df["total_xg_per90"][3] == 0 # Paul
assert test_results_df["total_xg_per90"][4] == 0 # Steve

# touches_in_box_per90
assert test_results_df["touches_in_box_per90"][0] == (1/500)*90 # Andy performed one carry into box
assert test_results_df["touches_in_box_per90"][1] == (2/1000)*90 # Chris took two shots in box
assert test_results_df["touches_in_box_per90"][2] == 0 # Pete
assert test_results_df["touches_in_box_per90"][3] == 0 # Paul
assert test_results_df["touches_in_box_per90"][4] == (1/500)*90 # Steve performed one carry into box

# aerial_duels_won_per90
assert test_results_df["aerial_duels_won_per90"][0] == (1/500)*90 # Andy should be the only one
assert test_results_df["aerial_duels_won_per90"][1] == 0 # Chris 
assert test_results_df["aerial_duels_won_per90"][2] == 0 # Pete
assert test_results_df["aerial_duels_won_per90"][3] == 0 # Paul
assert test_results_df["aerial_duels_won_per90"][4] == 0 # Steve

Generating Dummy Events
Passing through features engineering pipeline
Count of players with Event Stats: 5
Count of players with Minutes calculated: 5
Count of players after Merge: 5
Players with Stats but no minutes detected: 0
Players with > 300 Minutes: 5
                  total_passes  successful_passes  pass_completion_pct  \
Attacking Andy             1.0                0.0                  0.0   
Countering Chris          19.0               19.0                  1.0   
Possession Pete           80.0               80.0                  1.0   
Pressing Paul              0.0                0.0                  0.0   
Subbed-on Steve            0.0                0.0                  0.0   

                  passes_into_final_third  progressive_carries  \
Attacking Andy                        0.0                  0.0   
Countering Chris                      0.0                  0.0   
Possession Pete                       1.0                  1.0   
Pressing Paul                   

# Final Dataset Generation

In [124]:
# Get all free competitions and store into a master directory
df = sb.competitions()

# List to store finished player stats
all_processed_seasons = []

for x in selected_ids:
    # Search the master directory for the competition and season id's in the selected seasons list far above
    my_season = df[((df['competition_id'] == x['competition_id']) & (df['season_id'] == x['season_id']))]
    
    # Extract the country, competition and season names for each season in the selection list
    # Strip removes any potential invisible spaces or formatting
    country_name = my_season["country_name"].to_string(index=False).strip()
    competition_name = my_season["competition_name"].to_string(index=False).strip()
    season_name = my_season["season_name"].to_string(index=False).strip()
    
    print(f"Processing: {competition_name} {season_name}")
    
    # Fetch the raw events for this specific season
    events = sb.competition_events(country_name, competition_name, season_name)
    
    # Calculate the stats
    season_data = build_final_dataset(events)
    
    # Get player metadata
    metadata = get_player_metadata(events)
    
    # Inject competition and season into the metadata
    metadata["competition"] = competition_name
    metadata["season"] = season_name
    
    # Merge the stats and metadata
    final_season_data = season_data.join(metadata, how="left")
    
    # Add this finished season to the master list
    all_processed_seasons.append(final_season_data)

# Combine every processed season into one massive DataFrame
complete_combined_df = pd.concat(all_processed_seasons)

# Export the final combined file to a csv
complete_combined_df.to_csv(DATA_PATH / "Combined_Dataset_Final.csv")

print(f"\nSuccessfully generated Combined_Dataset_Final.csv with {len(complete_combined_df)} players!")

Processing: Serie A 2015/2016
Count of players with Event Stats: 550
Count of players with Minutes calculated: 551
Count of players after Merge: 551
Players with Stats but no minutes detected: 0
Players with > 300 Minutes: 455
Processing: Premier League 2015/2016
Count of players with Event Stats: 554
Count of players with Minutes calculated: 555
Count of players after Merge: 555
Players with Stats but no minutes detected: 0
Players with > 300 Minutes: 431
Processing: La Liga 2015/2016
Count of players with Event Stats: 538
Count of players with Minutes calculated: 539
Count of players after Merge: 539
Players with Stats but no minutes detected: 0
Players with > 300 Minutes: 450
Processing: Ligue 1 2015/2016
Count of players with Event Stats: 576
Count of players with Minutes calculated: 578
Count of players after Merge: 578
Players with Stats but no minutes detected: 0
Players with > 300 Minutes: 460
Processing: 1. Bundesliga 2015/2016
Count of players with Event Stats: 345
Count of p